# 07 · Sistema RAG Completo con LangGraph
Integra todos los componentes en un **grafo de ejecución** con LangGraph.
Las transiciones son predefinidas (no autónomas).

```
START
  │
  ▼
[clasificar]  ── (requiere_rag=True)  ──► [recuperar] ─► [generar] ─► [verificar]
  │                                                                         │
  └── (requiere_rag=False) ──► [respuesta_directa] ─► END          ┌──────┴──────┐
                                                              (pass) │             │ (fail, intento<3)
                                                                     ▼             ▼
                                                                    END       [generar]
```

### Asignación de LLMs
| Nodo | LLM | Justificación |
|---|---|---|
| `clasificar` | **Gemini** | Comprensión contextual profunda |
| `recuperar` (k dinámico) | **Groq** | Latencia ultra-baja para decisión simple |
| `generar` | **Gemini** | Síntesis de texto larga y coherente con citas |
| `verificar` | **Gemini** | Razonamiento crítico multi-criterio |
| `respuesta_directa` | **Groq** | Velocidad en preguntas simples sin corpus |

### Tools implementadas (≥5)
1. `buscar_documentos` — búsqueda semántica en FAISS
2. `resumir_documento` — resumen de un paper por doc_id
3. `comparar_documentos` — comparación de papers
4. `listar_documentos` — catálogo del corpus
5. `buscar_por_autor` — filtro por autor
6. `obtener_metadata` — metadatos de un paper


In [ ]:
# ── Instalar todas las dependencias ──────────────────────────────────────
!pip install -q \
    langchain==0.3.14 \
    langchain-community==0.3.14 \
    langchain-google-genai==2.0.8 \
    langchain-groq==0.2.3 \
    langgraph==0.2.70 \
    faiss-cpu==1.9.0 \
    pymupdf==1.25.2 \
    pydantic==2.10.4
print('✓ Todas las dependencias instaladas')


In [ ]:
# ── Montar Drive y configuración global ───────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json
from typing import TypedDict, Literal, Annotated
import operator

BASE_DIR  = '/content/drive/MyDrive/RAG_BioMed'
INDEX_DIR = f'{BASE_DIR}/faiss_index'
MANIFEST  = f'{BASE_DIR}/manifest.json'

GEMINI_KEY   = userdata.get('GEMINI_API_KEY')
GROQ_KEY     = userdata.get('GROQ_API_KEY')
GEMINI_MODEL = 'gemini-2.0-flash'
GROQ_MODEL   = 'llama-3.3-70b-versatile'
EMBED_MODEL  = 'models/text-embedding-004'
MAX_REINTENTOS = 3


In [ ]:
# ── Inicializar modelos y vectorstore ─────────────────────────────────────
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS

# Modelos
gemini      = ChatGoogleGenerativeAI(model=GEMINI_MODEL,
                                     google_api_key=GEMINI_KEY,
                                     temperature=0.2)
gemini_det  = ChatGoogleGenerativeAI(model=GEMINI_MODEL,
                                     google_api_key=GEMINI_KEY,
                                     temperature=0.0)
groq_llm    = ChatGroq(model=GROQ_MODEL, groq_api_key=GROQ_KEY, temperature=0.0)

# Vectorstore
_emb = GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, google_api_key=GEMINI_KEY)
vs   = FAISS.load_local(INDEX_DIR, _emb, allow_dangerous_deserialization=True)
print(f'✓ Vectorstore: {vs.index.ntotal} vectores')

# Manifiesto
with open(MANIFEST, 'r', encoding='utf-8') as f:
    _manifest = json.load(f)
manifest_idx = {d['doc_id']: d for d in _manifest}
print(f'✓ Manifiesto: {len(_manifest)} documentos')


In [ ]:
from langchain_core.tools import tool

# ═══════════════════════════════════════════════════════════════════════════
# TOOL 1: Búsqueda semántica
# ═══════════════════════════════════════════════════════════════════════════
@tool
def buscar_documentos(query: str, k: int = 5) -> str:
    """
    Realiza búsqueda semántica en el corpus biomédico (50 papers de arXiv).
    Retorna los fragmentos más relevantes con metadatos completos.

    Args:
        query : Consulta en lenguaje natural.
        k     : Número de fragmentos a recuperar (default 5).
    """
    docs = vs.similarity_search_with_score(query, k=k)
    out  = []
    for doc, score in docs:
        m = doc.metadata
        out.append({
            'doc_id'  : m.get('doc_id'),
            'titulo'  : m.get('titulo', '')[:70],
            'pagina'  : m.get('page', 0) + 1,
            'chunk_id': m.get('chunk_id'),
            'score'   : round(float(score), 4),
            'texto'   : doc.page_content[:400],
        })
    return json.dumps(out, ensure_ascii=False, indent=2)

# ═══════════════════════════════════════════════════════════════════════════
# TOOL 2: Resumir documento
# ═══════════════════════════════════════════════════════════════════════════
@tool
def resumir_documento(doc_id: str) -> str:
    """
    Resume el contenido de un paper específico del corpus dado su doc_id.
    Usa Gemini para sintetizar los fragmentos más representativos.

    Args:
        doc_id : Identificador del documento (e.g. 'doc_01').
    """
    info  = manifest_idx.get(doc_id, {})
    if not info:
        return f'Documento {doc_id} no encontrado en el corpus.'
    docs  = vs.similarity_search(
        f'main contributions methodology results of {info.get("titulo","")}',
        k=6, filter={'doc_id': doc_id}
    )
    if not docs:
        return f'No se encontraron fragmentos para {doc_id}.'
    ctx   = '\n---\n'.join(d.page_content for d in docs[:5])
    resp  = gemini.invoke(
        f'Resume el siguiente artículo científico de forma concisa (máx. 300 palabras).\n'
        f'Título: {info.get("titulo","")}\n\nContenido:\n{ctx[:4000]}'
    )
    return resp.content

# ═══════════════════════════════════════════════════════════════════════════
# TOOL 3: Comparar documentos
# ═══════════════════════════════════════════════════════════════════════════
@tool
def comparar_documentos(doc_ids: str, aspecto: str = 'metodología') -> str:
    """
    Compara dos o más papers del corpus respecto a un aspecto específico.

    Args:
        doc_ids : IDs separados por coma (e.g. 'doc_01,doc_03').
        aspecto : Dimensión de comparación (e.g. 'metodología', 'resultados').
    """
    ids = [i.strip() for i in doc_ids.split(',')]
    if len(ids) < 2:
        return 'Se necesitan al menos 2 doc_ids para comparar.'
    textos = {}
    for did in ids:
        docs = vs.similarity_search(aspecto, k=4, filter={'doc_id': did})
        if docs:
            titulo  = manifest_idx.get(did, {}).get('titulo', did)
            textos[did] = f'TÍTULO: {titulo}\n' + '\n'.join(d.page_content for d in docs[:3])
    if len(textos) < 2:
        return 'No se recuperaron suficientes fragmentos para la comparación.'
    ctx  = '\n\n===\n\n'.join(f'[{k}]:\n{v}' for k, v in textos.items())
    resp = gemini.invoke(
        f'Compara los siguientes papers respecto a "{aspecto}":\n\n{ctx[:5000]}'
    )
    return resp.content

# ═══════════════════════════════════════════════════════════════════════════
# TOOL 4: Listar documentos del corpus
# ═══════════════════════════════════════════════════════════════════════════
@tool
def listar_documentos(anio: int = 0) -> str:
    """
    Lista todos los documentos disponibles en el corpus.

    Args:
        anio : Filtrar por año de publicación (0 = todos los años).
    """
    docs = _manifest if anio == 0 else [d for d in _manifest if d.get('anio') == anio]
    if not docs:
        return f'No hay documentos del año {anio}.'
    lineas = [
        f'{d["doc_id"]}: {d["titulo"][:65]}  ({d["anio"]})  '
        f'— {d["autores"][0] if d["autores"] else "N/A"}'
        for d in docs
    ]
    return '\n'.join(lineas)

# ═══════════════════════════════════════════════════════════════════════════
# TOOL 5: Buscar por autor
# ═══════════════════════════════════════════════════════════════════════════
@tool
def buscar_por_autor(nombre: str) -> str:
    """
    Busca papers en el corpus cuyos autores contengan `nombre`.

    Args:
        nombre : Nombre o apellido del autor (búsqueda parcial, no sensible a mayúsculas).
    """
    hits = [
        d for d in _manifest
        if any(nombre.lower() in a.lower() for a in d.get('autores', []))
    ]
    if not hits:
        return f'No se encontraron papers de "{nombre}" en el corpus.'
    return '\n'.join(
        f'{d["doc_id"]}: {d["titulo"][:65]}  — {", ".join(d["autores"][:3])}'
        for d in hits
    )

# ═══════════════════════════════════════════════════════════════════════════
# TOOL 6: Obtener metadatos
# ═══════════════════════════════════════════════════════════════════════════
@tool
def obtener_metadata(doc_id: str) -> str:
    """
    Retorna los metadatos completos de un paper: título, autores, año,
    arxiv_id y abstract.

    Args:
        doc_id : Identificador del documento (e.g. 'doc_01').
    """
    info = manifest_idx.get(doc_id)
    if not info:
        return f'Documento {doc_id} no encontrado.'
    return json.dumps({
        'doc_id'   : info['doc_id'],
        'titulo'   : info['titulo'],
        'autores'  : info['autores'],
        'anio'     : info['anio'],
        'arxiv_id' : info['arxiv_id'],
        'abstract' : info.get('abstract', '')[:500],
    }, ensure_ascii=False, indent=2)

TOOLS = [buscar_documentos, resumir_documento, comparar_documentos,
         listar_documentos, buscar_por_autor, obtener_metadata]
print(f'✓ {len(TOOLS)} tools registradas: {[t.name for t in TOOLS]}')


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# ── Esquemas Pydantic para salida estructurada ─────────────────────────────

class ClasificacionConsulta(BaseModel):
    """Salida del nodo clasificador."""
    categoria: Literal['búsqueda', 'resumen', 'comparación', 'general']
    requiere_rag: bool
    doc_ids_mencionados: list[str] = []
    razonamiento: str

class ResultadoVerificacion(BaseModel):
    """Salida del nodo verificador."""
    aprobado: bool
    puntuacion_grounding: float = Field(ge=0.0, le=1.0)
    puntuacion_coherencia: float = Field(ge=0.0, le=1.0)
    puntuacion_completitud: float = Field(ge=0.0, le=1.0)
    problemas: list[str] = []
    sugerencias: str = ''


In [ ]:
# ── Estado del grafo ─────────────────────────────────────────────────────
class RAGState(TypedDict):
    query               : str
    tipo_consulta       : str   # búsqueda | resumen | comparación | general
    requiere_rag        : bool
    docs_recuperados    : list  # fragmentos serializados
    contexto            : str   # texto formateado para el prompt
    respuesta           : str
    verificacion        : dict
    intento             : int
    respuesta_final     : str
    traza               : dict  # registro completo de cada nodo


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.documents import Document

# ─────────────────────────────────────────────────────────────────────────
# NODO 1 · Clasificar (Gemini)
# ─────────────────────────────────────────────────────────────────────────
def nodo_clasificar(state: RAGState) -> RAGState:
    """
    Clasifica la intención de la consulta con Gemini.
    LLM elegido: Gemini (comprensión contextual profunda y razonamiento semántico).
    """
    parser = PydanticOutputParser(pydantic_object=ClasificacionConsulta)
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Clasifica consultas para un sistema RAG biomédico (50 papers arXiv).\n'
                   'Categorías: búsqueda, resumen, comparación, general.\n{format_instructions}'),
        ('human', 'Consulta: {query}')
    ])
    resultado = (prompt | gemini_det | parser).invoke({
        'query': state['query'],
        'format_instructions': parser.get_format_instructions()
    })
    return {
        **state,
        'tipo_consulta' : resultado.categoria,
        'requiere_rag'  : resultado.requiere_rag,
        'traza': {
            **state.get('traza', {}),
            'clasificacion': {
                'categoria'    : resultado.categoria,
                'requiere_rag' : resultado.requiere_rag,
                'razonamiento' : resultado.razonamiento,
                'llm'          : f'Gemini · {GEMINI_MODEL}',
            }
        }
    }

# ─────────────────────────────────────────────────────────────────────────
# NODO 2 · Recuperar (Groq para k + FAISS)
# ─────────────────────────────────────────────────────────────────────────
def nodo_recuperar(state: RAGState) -> RAGState:
    """
    Determina k dinámicamente con Groq y ejecuta la búsqueda semántica.
    LLM elegido: Groq (latencia ultra-baja para decisión simple de k).
    """
    # k dinámico con Groq
    k_resp = groq_llm.invoke(
        f'¿Cuántos fragmentos (3-12) necesito para responder una consulta '
        f'de tipo "{state["tipo_consulta"]}"? Consulta: "{state["query"][:80]}"\n'
        f'Responde SOLO el número entero.'
    )
    try:
        k = max(3, min(int(k_resp.content.strip()), 12))
    except ValueError:
        k = 5

    docs_scores = vs.similarity_search_with_score(state['query'], k=k)

    # Serializar documentos
    docs_ser, partes = [], []
    for i, (doc, score) in enumerate(docs_scores, 1):
        m = doc.metadata
        docs_ser.append({
            'contenido': doc.page_content,
            'metadata' : {
                'doc_id'          : m.get('doc_id'),
                'titulo'          : m.get('titulo', '')[:70],
                'autores'         : m.get('autores', ''),
                'pagina'          : m.get('page', 0) + 1,
                'chunk_id'        : m.get('chunk_id', ''),
                'similarity_score': round(float(score), 4),
            }
        })
        meta_str = (f'[{i}] {m.get("titulo","")[:55]} | '
                    f'{m.get("doc_id")} p.{m.get("page",0)+1} | '
                    f'chunk={m.get("chunk_id","")}')
        partes.append(f'--- Fragmento {i} ---\n{meta_str}\n{doc.page_content}')

    return {
        **state,
        'docs_recuperados': docs_ser,
        'contexto'        : '\n\n'.join(partes),
        'traza': {
            **state.get('traza', {}),
            'recuperacion': {
                'k_dinamico': k,
                'num_docs'  : len(docs_ser),
                'docs'      : [f'{d["metadata"]["doc_id"]} (score={d["metadata"]["similarity_score"]})'
                               for d in docs_ser],
                'llm_k'     : f'Groq · {GROQ_MODEL}',
            }
        }
    }

# ─────────────────────────────────────────────────────────────────────────
# NODO 3 · Generar respuesta (Gemini)
# ─────────────────────────────────────────────────────────────────────────
def nodo_generar(state: RAGState) -> RAGState:
    """
    Genera la respuesta basada en contexto con citas numeradas.
    LLM elegido: Gemini (síntesis larga, coherente, con instrucciones de formato).
    """
    sugerencias = state.get('verificacion', {}).get('sugerencias', '')
    extra       = f'\n\nMEJORA SOLICITADA: {sugerencias}' if sugerencias else ''

    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Eres un experto en bioinformática basado en 50 papers científicos de arXiv.\n'
                   'Responde SOLO usando el contexto. Incluye citas [1],[2]...\n'
                   'Si no está en el contexto escribe: "No encontrado en el corpus".\n\n'
                   'CONTEXTO:\n{contexto}'),
        ('human', '{query}')
    ])
    resp = (prompt | gemini).invoke({
        'query'  : state['query'] + extra,
        'contexto': state['contexto'][:6000]
    })
    return {
        **state,
        'respuesta': resp.content,
        'traza': {
            **state.get('traza', {}),
            'generacion': {
                'intento'        : state.get('intento', 1),
                'llm'            : f'Gemini · {GEMINI_MODEL}',
                'longitud_resp'  : len(resp.content),
                'con_sugerencias': bool(sugerencias),
            }
        }
    }

# ─────────────────────────────────────────────────────────────────────────
# NODO 4 · Verificar (Gemini)
# ─────────────────────────────────────────────────────────────────────────
def nodo_verificar(state: RAGState) -> RAGState:
    """
    Verifica grounding, coherencia y completitud con Gemini.
    LLM elegido: Gemini (razonamiento crítico multi-criterio).
    """
    parser = PydanticOutputParser(pydantic_object=ResultadoVerificacion)
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Verifica la respuesta RAG. Puntúa 0-1: grounding, coherencia, completitud.\n'
                   'aprobado=True si los tres >= 0.7.\n{format_instructions}\n\nCONTEXTO:\n{contexto}'),
        ('human', 'PREGUNTA:\n{query}\n\nRESPUESTA:\n{respuesta}')
    ])
    ver = (prompt | gemini_det | parser).invoke({
        'query'              : state['query'],
        'respuesta'          : state['respuesta'],
        'contexto'           : state['contexto'][:3000],
        'format_instructions': parser.get_format_instructions()
    })
    ver_dict = {
        'aprobado'    : ver.aprobado,
        'grounding'   : ver.puntuacion_grounding,
        'coherencia'  : ver.puntuacion_coherencia,
        'completitud' : ver.puntuacion_completitud,
        'problemas'   : ver.problemas,
        'sugerencias' : ver.sugerencias,
    }
    nuevo_estado = {
        **state,
        'verificacion'   : ver_dict,
        'intento'        : state.get('intento', 1),
        'respuesta_final': state['respuesta'] if ver.aprobado else state.get('respuesta_final', ''),
        'traza': {
            **state.get('traza', {}),
            'verificacion': {**ver_dict, 'llm': f'Gemini · {GEMINI_MODEL}'}
        }
    }
    return nuevo_estado

# ─────────────────────────────────────────────────────────────────────────
# NODO 5 · Respuesta directa (Groq)
# ─────────────────────────────────────────────────────────────────────────
def nodo_respuesta_directa(state: RAGState) -> RAGState:
    """
    Responde preguntas generales sin acceder al corpus.
    LLM elegido: Groq (velocidad para preguntas simples sin necesidad de contexto).
    """
    resp = groq_llm.invoke(state['query'])
    return {
        **state,
        'respuesta'      : resp.content,
        'respuesta_final': resp.content,
        'traza': {
            **state.get('traza', {}),
            'respuesta_directa': {
                'llm'    : f'Groq · {GROQ_MODEL}',
                'sin_rag': True,
            }
        }
    }


In [ ]:
from langgraph.graph import StateGraph, START, END

# ── Enrutadores condicionales ──────────────────────────────────────────────

def enrutar_clasificacion(state: RAGState) -> str:
    """Enruta a recuperación (RAG) o respuesta directa según clasificación."""
    return 'recuperar' if state['requiere_rag'] else 'respuesta_directa'

def enrutar_verificacion(state: RAGState) -> str:
    """
    Decide si la respuesta fue aprobada o se necesita regenerar.
    Limita a MAX_REINTENTOS para evitar loops infinitos.
    """
    if state['verificacion']['aprobado']:
        return 'aprobado'
    intento_actual = state.get('intento', 1)
    if intento_actual >= MAX_REINTENTOS:
        return 'aprobado'   # forzar salida tras máximo de intentos
    state['intento'] = intento_actual + 1
    return 'regenerar'

# ── Construcción del grafo ─────────────────────────────────────────────────
def construir_grafo() -> any:
    """
    Construye y compila el grafo LangGraph del sistema RAG.

    Nodos:
        clasificar, recuperar, generar, verificar,
        respuesta_directa

    Transiciones predefinidas (no autónomas):
        START → clasificar
        clasificar → [recuperar | respuesta_directa]  (condicional)
        recuperar  → generar
        generar    → verificar
        verificar  → [END | generar]                  (condicional)
        respuesta_directa → END
    """
    wf = StateGraph(RAGState)

    # Registrar nodos
    wf.add_node('clasificar',         nodo_clasificar)
    wf.add_node('recuperar',          nodo_recuperar)
    wf.add_node('generar',            nodo_generar)
    wf.add_node('verificar',          nodo_verificar)
    wf.add_node('respuesta_directa',  nodo_respuesta_directa)

    # Aristas fijas
    wf.add_edge(START,       'clasificar')
    wf.add_edge('recuperar', 'generar')
    wf.add_edge('generar',   'verificar')
    wf.add_edge('respuesta_directa', END)

    # Aristas condicionales
    wf.add_conditional_edges(
        'clasificar', enrutar_clasificacion,
        {'recuperar': 'recuperar', 'respuesta_directa': 'respuesta_directa'}
    )
    wf.add_conditional_edges(
        'verificar', enrutar_verificacion,
        {'aprobado': END, 'regenerar': 'generar'}
    )

    return wf.compile()

app = construir_grafo()
print('✓ Grafo LangGraph compilado correctamente')


In [ ]:
# ── Visualizar el grafo ───────────────────────────────────────────────────
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'Visualización no disponible: {e}')
    print('Descripción del flujo:')
    print('  START → clasificar')
    print('  clasificar → recuperar (si requiere_rag)')
    print('  clasificar → respuesta_directa (si general)')
    print('  recuperar  → generar → verificar')
    print('  verificar  → END (si aprobado) | generar (si falla, máx. 3 veces)')
    print('  respuesta_directa → END')


In [ ]:
def ejecutar_rag(query: str, verbose: bool = True) -> dict:
    """
    Punto de entrada principal del sistema RAG.

    Inicializa el estado y ejecuta el grafo completo. Retorna el estado
    final con la respuesta, citas, trazabilidad y resultado de verificación.

    Args:
        query   : Pregunta del usuario en lenguaje natural.
        verbose : Si True, imprime la trazabilidad completa.

    Returns:
        Estado final del grafo (RAGState dict).
    """
    estado_inicial: RAGState = {
        'query'            : query,
        'tipo_consulta'    : '',
        'requiere_rag'     : True,
        'docs_recuperados' : [],
        'contexto'         : '',
        'respuesta'        : '',
        'verificacion'     : {},
        'intento'          : 1,
        'respuesta_final'  : '',
        'traza'            : {},
    }

    resultado = app.invoke(estado_inicial)

    if verbose:
        sep = '═' * 65
        print(f'\n{sep}')
        print('TRAZABILIDAD COMPLETA')
        print(sep)
        for nodo, info in resultado['traza'].items():
            print(f'\n▶ {nodo.upper()}')
            for k, v in info.items():
                val = str(v)[:120] + ('...' if len(str(v)) > 120 else '')
                print(f'    {k:20s}: {val}')

        if resultado.get('docs_recuperados'):
            print(f'\n{sep}')
            print('DOCUMENTOS RECUPERADOS')
            print(sep)
            for d in resultado['docs_recuperados']:
                m = d['metadata']
                print(f'  • {m["doc_id"]} | p.{m["pagina"]} | '
                      f'{m["titulo"][:50]}  score={m["similarity_score"]}')

        print(f'\n{sep}')
        print('RESPUESTA FINAL')
        print(sep)
        print(resultado['respuesta_final'])

    return resultado


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CASOS DE USO — 10 consultas de demostración
# ═══════════════════════════════════════════════════════════════════════════

CASOS_DE_USO = [
    # Búsquedas semánticas
    ('CU-01', '¿Qué métodos de deep learning se usan para predecir la estructura de proteínas?'),
    ('CU-02', '¿Cuáles son los principales datasets utilizados para diagnóstico de cáncer con ML?'),
    ('CU-03', '¿Cómo se aplican las redes convolucionales (CNN) al análisis de imágenes médicas?'),
    ('CU-04', '¿Qué técnicas de NLP se usan para extraer información de registros clínicos?'),
    # Resúmenes
    ('CU-05', 'Resume el paper doc_01'),
    ('CU-06', 'Dame un resumen de los papers sobre drug discovery del corpus'),
    # Comparaciones
    ('CU-07', 'Compara los enfoques de doc_02 y doc_04 en términos de metodología'),
    # Consulta general
    ('CU-08', '¿Qué es el aprendizaje por transferencia (transfer learning)?'),
    # Búsqueda específica
    ('CU-09', '¿Qué modelos de lenguaje se han aplicado a secuencias genómicas?'),
    ('CU-10', 'Lista los papers del corpus publicados en 2023'),
]

# Ejecutar primer caso de uso como demostración
print('Ejecutando CU-01 como demostración...')
print('(Para ejecutar todos, descomenta el bucle al final)\n')

cu_id, consulta = CASOS_DE_USO[0]
print(f'[{cu_id}] {consulta}')
resultado_demo = ejecutar_rag(consulta)

# ── Para ejecutar todos los casos (puede tardar varios minutos) ────────────
# resultados_todos = {}
# for cu_id, consulta in CASOS_DE_USO:
#     print(f'\n{'='*60}')
#     print(f'Caso: {cu_id}')
#     resultados_todos[cu_id] = ejecutar_rag(consulta)


In [ ]:
# ── Uso de tools de forma directa ─────────────────────────────────────────
print('=== USO DIRECTO DE TOOLS ===\n')

print('TOOL 1 · buscar_documentos:')
print(buscar_documentos.invoke({'query': 'protein folding neural network', 'k': 3}))

print('\nTOOL 4 · listar_documentos:')
print(listar_documentos.invoke({'anio': 0})[:500], '...')

print('\nTOOL 5 · buscar_por_autor:')
print(buscar_por_autor.invoke({'nombre': 'Li'}))
